In [1]:
!pip install pyspark


In [5]:
import sqlite3, json, pandas as pd

df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)
print(df.shape)
df.head()

(2000, 7)


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01,8,Gasabo,send,1000,0
1,T00002,2026-11-09,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19,8,Muhanga,send,51000,0
3,T00004,2026-11-01,17,Huye,send,79000,0
4,T00005,2026-11-15,12,Huye,send,109000,0


In [20]:
## Exercises Part 1.1 
# (a) 5 largest 'send' transactions above 100,000 RWF

q =""" 
SELECT txn_id , district , amount_rwf 
FROM transactions 
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5
"""
pd.read_sql(q,conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


In [7]:
# (b) flagged transactions  (checkpoint: 11)

pd.read_sql ("SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged =1",conn) 

,flagged_count
0,11


In [21]:
# (a) per txn_type  (checkpoint: send ~62.0M)
q_type = """
SELECT txn_type, 
       COUNT(*) AS n,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
"""
print(pd.read_sql(q_type, conn))

  txn_type    n  total_rwf       avg_rwf
0     send  868   62049000  71485.023041
1  paybill  330   29970000  90818.181818
2  receive  489   27024000  55263.803681
3  airtime  313     833000   2661.341853


In [17]:
# (b) per district  (checkpoint: Muhanga ~21.1M)

q_dist = """
SELECT district, 
       SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC
"""
print (pd.read_sql(q_dist, conn))

     district  total_rwf
0     Muhanga   21076000
1      Gasabo   20738000
2        Huye   18736000
3  Nyarugenge   18091000
4    Kicukiro   11338000
5     Musanze   10490500
6      Rubavu   10110500
7   Nyagatare    9296000


In [3]:
# Exercises 1.3
# (a) SQL  (checkpoint: hour 20, 185) 
sql_hours = pd.read_sql (""" SELECT hour,COUNT(*) AS n 
                             FROM transactions 
                             GROUP BY hour 
                             ORDER BY n DESC """,conn)
sql_hours.head()

,hour,n
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169


In [20]:
 ## b) 1.3

pd_hours = df['hour'].value_counts()
print(pd_hours.head())
print('Agree:', pd_hours.index[0] == sql_hours.loc[0,'hour'] and pd_hours.iloc[0] == sql_hours.loc[0,'n'])

hour
20    185
17    183
18    176
12    170
11    169
Name: count, dtype: int64
Agree: True


In [7]:

## Part 2 · APIs & JSON


with open('week12_api_sample.json') as f:
    data = json.load(f)

temps = data['hourly']['temperature_2m']
times = data['hourly']['time']

print('(a) Hours:', len(temps))
print('(b) Max:', max(temps), '  Min:', min(temps))
print('(c) Hottest hour at:', times[temps.index(max(temps))])

(a) Hours: 24
(b) Max: 27.9   Min: 15.9
(c) Hottest hour at: 2026-11-22T14:00


In [8]:
wx = pd.DataFrame({'time': times, 'temp_c': temps})
print(wx['temp_c'].describe())                              # (a)
print('(b) Hours >= 25 C:', (wx['temp_c'] >= 25).sum())     # (b)

count    24.000000
mean     21.137500
std       4.104696
min      15.900000
25%      17.650000
50%      19.900000
75%      25.025000
max      27.900000
Name: temp_c, dtype: float64
(b) Hours >= 25 C: 7


In [12]:
## Part 3 · PySpark
### Exercise 3.1 — Load & inspect

from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)
print('Rows:', sdf.count())
sdf.printSchema()
sdf.show(5)


Rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


In [13]:

print('(a) > 100k:', sdf.filter(sdf.amount_rwf > 100000).count())   # checkpoint 451
sdf.filter(sdf.flagged == 1).show(5) 

(a) > 100k: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


In [14]:
sdf.groupBy('txn_type').sum('amount_rwf').orderBy('sum(amount_rwf)', ascending=False).show() 

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
|    send|       62049000|
| paybill|       29970000|
| receive|       27024000|
| airtime|         833000|
+--------+---------------+



In [15]:
dist_pd = sdf.groupBy('district').sum('amount_rwf').toPandas()                                  # (b)
dist_pd.sort_values('sum(amount_rwf)', ascending=False)

,district,sum(amount_rwf)
4,Muhanga,21076000
5,Gasabo,20738000
6,Huye,18736000
1,Nyarugenge,18091000
0,Kicukiro,11338000
2,Musanze,10490500
7,Rubavu,10110500
3,Nyagatare,9296000


In [18]:
# (c) Do the Spark answers match SQL?
# Yes. Spark gives send = 62,049,000 RWF as the top type and Muhanga = 21,076,000 RWF as the top district,
# identical to the SQL results. Same data, same aggregation, so the answers must agree.
sql_d = pd.read_sql(q_dist, conn).set_index('district')['total_rwf']
spk_d = dist_pd.set_index('district')['sum(amount_rwf)']
print('District totals identical:', sql_d.sort_index().equals(spk_d.sort_index().astype(sql_d.dtype)))

District totals identical: True


In [22]:
### Bonus — Spark speaks SQL too

sdf.createOrReplaceTempView('transactions')
spark.sql(q_type).show()   # the Part 1 query, unchanged
# Why powerful: SQL is one portable language. The query you test on a small SQLite file runs unchanged on a
# cluster of thousands of machines; the engine handles distribution, so you don't rewrite logic or learn a new API to scale.

+--------+---+---------+------------------+
|txn_type|  n|total_rwf|           avg_rwf|
+--------+---+---------+------------------+
|    send|868| 62049000| 71485.02304147465|
| paybill|330| 29970000| 90818.18181818182|
| receive|489| 27024000|  55263.8036809816|
| airtime|313|   833000|2661.3418530351437|
+--------+---+---------+------------------+



In [ ]:
## Part 4 · Choose Your Weapon

1. **SQL** — the data already lives in PostgreSQL, so push a `GROUP BY district, month` to the database instead of moving data out.
2. **Pandas** — 40,000 rows fit easily in memory, and Pandas plus matplotlib/scikit-learn covers cleaning, charts and regression in one place.
3. **Spark** — 40 TB on a cluster is far beyond one machine, so use distributed processing where the data already sits.
4. **API** — only an API gives live, current weather on request with no manual file uploads.